# Fase 4b — Tríos (Pregunta, Positiva, Negativa) para Triplet Loss (TRAIN)

Toma `preguntas_train_expandido.json` (Fase 4a) y construye los tríos `(pregunta, positiva, negativa)` necesarios para afinar el modelo de embeddings con *triplet loss* (Fase 5b).

Para cada pregunta:

1. Se vectorizan todas las preguntas y todas las citas con el modelo E5 multilingüe (`intfloat/multilingual-e5-base`), respetando los prefijos `query:`/`passage:` que exige este modelo.
2. Se calcula la matriz completa de similitud coseno entre preguntas y citas.
3. Se descartan como candidatas a negativo todas las citas que provengan del **mismo macro-contexto** que la positiva (cortafuegos por `macro_id`), para no penalizar por error una respuesta igualmente válida.
4. Entre las 20 candidatas restantes más similares, se descartan también las que solapan léxicamente más de un 90% (`thefuzz`) con la cita positiva, evitando así "falsos negativos" casi idénticos al positivo.
5. La cita superviviente con mayor similitud se elige como **hard negative**: el negativo "más difícil" (semánticamente parecido pero realmente incorrecto), que es el que aporta más señal de entrenamiento.

El resultado se guarda en `dataset_train_triplets.json`. La primera pregunta procesada se imprime con detalle (ranking de candidatas y motivo de selección) a modo de auditoría visual del criterio aplicado.

In [3]:
import json
import torch
from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz

ARCHIVO_ENTRADA = 'preguntas_train_expandido.json'
ARCHIVO_SALIDA = 'dataset_train_triplets.json'
MODELO_NOMBRE = 'intfloat/multilingual-e5-base'

def generar_trios():
    """
    Construye tríos (pregunta, positiva, negativa) para el fine-tuning con
    triplet loss: vectoriza preguntas y citas con el modelo E5 multilingüe,
    calcula la similitud coseno entre todas las preguntas y todas las citas,
    y selecciona como 'hard negative' la cita de mayor similitud que NO
    pertenezca al mismo macro-contexto y que, además, no solape léxicamente
    (>90% en fuzzy matching) con la cita positiva real, para evitar falsos
    negativos (citas casi idénticas a la positiva que en realidad también
    serían una respuesta válida).
    """
    print("="*70)
    print(" INICIANDO EXTRACCIÓN DE HARD NEGATIVES (FASE 3) ".center(70))
    print("="*70)

    # 1. Cargar datos
    with open(ARCHIVO_ENTRADA, 'r', encoding='utf-8') as f:
        datos = json.load(f)

    print(f"-> Cargados {len(datos)} pares de entrenamiento.")
    print(f"-> Cargando modelo '{MODELO_NOMBRE}' ...")
    
    # Si tienes GPU (CUDA) lo usará automáticamente, si no, irá por CPU
    modelo = SentenceTransformer(MODELO_NOMBRE)

    # 2. Preparar los textos con los prefijos OBLIGATORIOS para el modelo E5
    # Extraemos solo los textos únicos para no calcular cosas repetidas
    queries = [f"query: {item['pregunta']}" for item in datos]
    passages = [f"passage: {item['cita_literal']}" for item in datos]
    macro_ids = [item['macro_id_origen'] for item in datos]

    print("-> Vectorizando Documentos (Passages)...")
    embeddings_passages = modelo.encode(passages, convert_to_tensor=True, show_progress_bar=True)
    
    print("-> Vectorizando Preguntas (Queries)...")
    embeddings_queries = modelo.encode(queries, convert_to_tensor=True, show_progress_bar=True)

    # 3. Calcular matriz de similitud del Coseno (Todas las preguntas vs Todas las citas)
    print("-> Calculando similitudes semánticas...")
    similitudes = util.cos_sim(embeddings_queries, embeddings_passages)

    trios = []
    falsos_negativos_evitados = 0

    # 4. Seleccionar el Hard Negative para cada pregunta
    for i in range(len(datos)):
        pregunta_actual = datos[i]['pregunta']
        positiva_actual = datos[i]['cita_literal']
        macro_id_actual = datos[i]['macro_id_origen']

        # Clonamos las puntuaciones de esta pregunta para no modificar la matriz original
        puntuaciones_pregunta = similitudes[i].clone()
        score_positiva_original = puntuaciones_pregunta[i].item() # Guardamos la nota real de la positiva

        # Cortafuegos 1 (Ultra rápido): Penalizamos por ID a todos de golpe
        for j in range(len(datos)):
            if macro_ids[j] == macro_id_actual:
                puntuaciones_pregunta[j] = -1.0 

        # OPTIMIZACIÓN: Solo le pasamos el thefuzz a los 20 mejores candidatos
        # No perdemos tiempo analizando textos que tienen un coseno bajo
        valores_top_preliminar, indices_top_preliminar = torch.topk(puntuaciones_pregunta, k=20)
        
        for idx in indices_top_preliminar:
            j = idx.item()
            if puntuaciones_pregunta[j] == -1.0:
                continue # Ya lo descartamos en el paso 1
                
            candidata_texto = datos[j]['cita_literal']
            
            # Cortafuegos 2: Solapamiento Léxico (Solo al Top 20)
            similitud_lexica = fuzz.partial_ratio(positiva_actual.lower(), candidata_texto.lower())
            
            if similitud_lexica > 90:
                puntuaciones_pregunta[j] = -1.0 # Destruimos la trampa
                falsos_negativos_evitados += 1

        # --- INICIO BLOQUE DE VALIDACIÓN (Solo para la Pregunta 1) ---
        if i == 0:
            print("="*70)
            print(" AUDITORÍA DE EXTRACCIÓN (EJEMPLO 1) ".center(70))
            print("="*70)
            print(f"[Q] Pregunta: {pregunta_actual}")
            print(f"[+] Score Positiva Original: {score_positiva_original:.4f} (Se ha forzado a -1.0 por ser del mismo doc)")
            
            # Sacamos el Top 3 de lo que ha quedado vivo tras el cortafuegos
            valores_top, indices_top = torch.topk(puntuaciones_pregunta, k=3)
            
            print("\n[RANKING DE TRAMPAS - DISTINTO MACRO_ID]:")
            for rank in range(3):
                idx = indices_top[rank].item()
                val = valores_top[rank].item()
                texto_muestra = datos[idx]['cita_literal'][:70].replace('\n', ' ')
                print(f"  {rank+1}º -> Score: {val:.4f} | ID Doc: {datos[idx]['macro_id_origen'][:8]}... | Texto: {texto_muestra}...")
            
            print(f"\n[!] SELECCIONADO COMO HARD NEGATIVE: El 1º (Score: {valores_top[0].item():.4f})")
            print("="*70 + "\n")
        # --- FIN BLOQUE DE VALIDACIÓN ---

        # Encontramos el índice de la cita con mayor puntuación restante (El Hard Negative)
        indice_hard_negative = torch.argmax(puntuaciones_pregunta).item()
        
        # Extraemos el texto crudo del Hard Negative (sin el prefijo 'passage: ')
        negativa_actual = datos[indice_hard_negative]['cita_literal']

        # Guardamos el Trío
        trios.append({
            "pregunta": pregunta_actual,
            "positiva": positiva_actual,
            "negativa": negativa_actual,
            "tipo": datos[i]['tipo'] # Mantenemos el metadato por si acaso
        })

    # 5. Guardar el nuevo dataset
    with open(ARCHIVO_SALIDA, 'w', encoding='utf-8') as f:
        json.dump(trios, f, ensure_ascii=False, indent=4)

    print(f"¡Extracción completada con éxito!")
    print(f" -> Se han generado {len(trios)} tríos de entrenamiento (Pregunta, Positiva, Negativa).")
    print(f" -> Falsos Negativos evitados por solapamiento: {falsos_negativos_evitados}")
    print(f" -> Guardado en: {ARCHIVO_SALIDA}")
    print("="*70)

if __name__ == "__main__":
    generar_trios()

           INICIANDO EXTRACCIÓN DE HARD NEGATIVES (FASE 3)            
-> Cargados 1015 pares de entrenamiento.
-> Cargando modelo 'intfloat/multilingual-e5-base' ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Vectorizando Documentos (Passages)...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

-> Vectorizando Preguntas (Queries)...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

-> Calculando similitudes semánticas...
                 AUDITORÍA DE EXTRACCIÓN (EJEMPLO 1)                  
[Q] Pregunta: ¿Qué medida no farmacológica se debe considerar ante una presentación aguda que genere dudas diagnósticas con una torsión testicular?
[+] Score Positiva Original: 0.8451 (Se ha forzado a -1.0 por ser del mismo doc)

[RANKING DE TRAMPAS - DISTINTO MACRO_ID]:
  1º -> Score: 0.8381 | ID Doc: 7a1cd7c0... | Texto: La purulencia del esputo no es indicación de tratamiento antibiótico e...
  2º -> Score: 0.8370 | ID Doc: a446c98a... | Texto: Se define como diverticulitis aguda no complicada aquella en la que só...
  3º -> Score: 0.8365 | ID Doc: f1f02f8b... | Texto: En caso de sospecha de congelación bien al recibir el envío o bien dur...

[!] SELECCIONADO COMO HARD NEGATIVE: El 1º (Score: 0.8381)

¡Extracción completada con éxito!
 -> Se han generado 1015 tríos de entrenamiento (Pregunta, Positiva, Negativa).
 -> Falsos Negativos evitados por solapamiento: 112
 -> Guard